# Load the registered model from Unity Catalog and predict on new data

In [0]:
import mlflow
from pyspark.sql.functions import col, round

In [0]:
# Load the registered model from Unity Catalog and predict on new data

print("🔄 Starting Batch Inference Pipeline...")

# Define Unity Catalog Volume path for temporary files
tmp_vol_path = "/Volumes/workspace/insurance_claim/mlflow_tmp"

## Load the Model from Unity Catalog

In [0]:
# Load the Model from Unity Catalog

# Use your registered model name and specify the version (e.g., 1 for the first time)
model_name = "workspace.insurance_claim.fraud_gbt_model"
model_version = 1 
model_uri = f"models:/{model_name}/{model_version}"

print(f"📥 Loading model from: {model_uri}")

loaded_model = mlflow.spark.load_model(
    model_uri=model_uri,
    dfs_tmpdir=tmp_vol_path 
)

In [0]:
#  Load "New" Unseen Data

# In a real-world scenario, you would load today's new claims data here.
# For this test, we are using the test_set to simulate new incoming data.
print("📂 Loading new claims data...")
new_data = spark.table("workspace.insurance_claim.test_set") 

In [0]:
# Make Predictions

print("🎯 Generating fraud predictions...")
predictions = loaded_model.transform(new_data)

In [0]:
# Format the Output for the Business/Investigation Team

# Select only the essential columns needed by the fraud investigators.
# Note: You should include your actual identifier columns here (e.g., "ClaimID" or "Provider").
final_results = predictions.select(
    col("is_fraud").alias("actual_label"),      # Keep this only if you want to compare results
    col("prediction").alias("predicted_fraud"), # 0.0 = Normal Claim, 1.0 = Suspicious/Fraud
    col("probability")                          # The model's confidence score
)

print("✅ Predictions generated successfully. Here is a sample:")
display(final_results.limit(10))

In [0]:
# Save Results to Unity Catalog

# Save the results to a new table so the investigation team can connect via SQL or BI tools (e.g., PowerBI).
output_table = "main.insurance_claim.daily_fraud_predictions"
final_results.write.mode("overwrite").saveAsTable(output_table)

print(f"💾 Saved predictions to: {output_table}")